#### Bibliotecas

In [42]:
import pandas as pd
import numpy as np
import os
import urllib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

#### FASE 0: Conexão com o Banco de Dados Azure e Carregamento dos Dados

In [43]:
def conectar_banco_azure():
    """
    Carrega as credenciais do arquivo .env e cria uma engine de conexão com o banco de dados Azure.
    """
    load_dotenv() # Carrega as variáveis de ambiente do arquivo .env

    db_server = os.getenv("DB_SERVER")
    db_database = os.getenv("DB_DATABASE")
    db_username = os.getenv("DB_USERNAME")
    db_password = os.getenv("DB_PASSWORD")

    if not all([db_server, db_database, db_username, db_password]):
        print("ERRO: Verifique se todas as variáveis de ambiente (DB_SERVER, DB_DATABASE, DB_USERNAME, DB_PASSWORD) estão no arquivo .env")
        return None

    try:
        params = urllib.parse.quote_plus(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"
            f"SERVER={db_server};"
            f"DATABASE={db_database};"
            f"UID={db_username};"
            f"PWD={db_password}"
        )
        engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
        print("Conexão com o banco de dados Azure bem-sucedida!")
        return engine
    except Exception as e:
        print(f"ERRO ao conectar ao banco de dados: {e}")
        return None

In [44]:
def carregar_dados_do_banco(engine):
    """
    Lê todas as tabelas necessárias do banco de dados para DataFrames do Pandas.
    """
    if engine is None:
        return None

    tabelas = [
        "Fato_Consumo", "Dim_Cliente", "Dim_Contrato", "Dim_Produto", "Dim_Nps",
        "Dim_Segmento", "Dim_Faturamento", "Dim_Localidade", "Dim_Modalidade",
        "Dim_StatusContrato", "Dim_Marca", "Dim_LinhaReceita"
    ]
    
    dataframes = {}
    print("\nCarregando tabelas do banco de dados...")
    try:
        with engine.connect() as conn:
            for tabela in tabelas:
                print(f"- Carregando {tabela}...")
                query = f"SELECT * FROM {tabela}"
                dataframes[tabela] = pd.read_sql(query, conn)
        print("Todas as tabelas foram carregadas com sucesso.")
        return dataframes
    except Exception as e:
        print(f"ERRO ao carregar tabelas: {e}")
        return None

In [22]:
# Conectar e carregar os dados
engine = conectar_banco_azure()
dfs = carregar_dados_do_banco(engine)

if dfs is None:
    print("Processo interrompido devido a erro no carregamento dos dados.")
    exit()

Conexão com o banco de dados Azure bem-sucedida!

Carregando tabelas do banco de dados...
- Carregando Fato_Consumo...
- Carregando Dim_Cliente...
- Carregando Dim_Contrato...
- Carregando Dim_Produto...
- Carregando Dim_Nps...
- Carregando Dim_Segmento...
- Carregando Dim_Faturamento...
- Carregando Dim_Localidade...
- Carregando Dim_Modalidade...
- Carregando Dim_StatusContrato...
- Carregando Dim_Marca...
- Carregando Dim_LinhaReceita...
Todas as tabelas foram carregadas com sucesso.


#### FASE 1: Consolidação e Preparação dos Dados

In [45]:
print("\nFASE 1: Consolidando e preparando os dados...")

# Juntar Fato_Consumo com informações do contrato para enriquecer os dados antes de agregar
consumo_enriquecido = pd.merge(dfs['Fato_Consumo'], dfs['Dim_Contrato'], on='cd_contrato', how='left')
consumo_enriquecido = pd.merge(consumo_enriquecido, dfs['Dim_StatusContrato'], on='cd_status', how='left')
consumo_enriquecido = pd.merge(consumo_enriquecido, dfs['Dim_Modalidade'], on='cd_modalidade', how='left')

# Agregar dados de consumo por cliente
consumo_agregado = consumo_enriquecido.groupby('cd_cliente').agg(
    vl_total_gasto=('vl_total', 'sum'),
    vl_medio_gasto=('vl_total', 'mean'),
    vl_total_desconto=('vl_desconto', 'sum'),
    qtd_compras=('cd_produto', 'count'),
    situacao_contrato=('situacao_contrato', lambda x: x.mode()[0] if not x.mode().empty else 'N/A'),
    modal_comerc=('modal_comerc', lambda x: x.mode()[0] if not x.mode().empty else 'N/A')
).reset_index()


FASE 1: Consolidando e preparando os dados...


In [46]:
# Para a tabela de NPS, vamos pegar a resposta mais recente de cada cliente
dfs['Dim_Nps']['respondeAt'] = pd.to_datetime(dfs['Dim_Nps']['respondeAt'])
nps_recente = dfs['Dim_Nps'].sort_values('respondeAt').drop_duplicates('cd_cliente', keep='last')

In [47]:
# A base agora são os clientes que efetivamente tiveram consumo.
df = pd.merge(consumo_agregado, dfs['Dim_Cliente'], on='cd_cliente', how='inner')

In [48]:
# Agora, enriquecemos essa base com os dados de NPS e as descrições
df = pd.merge(df, nps_recente, on='cd_cliente', how='left')
df = pd.merge(df, dfs['Dim_Segmento'], on='cd_segmento', how='left')
df = pd.merge(df, dfs['Dim_Faturamento'], on='cd_faturamento', how='left')

In [49]:
# Lidar com dados faltantes (estratégia aprimorada para evitar warnings)
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]):
        df[col] = df[col].fillna('N/A')
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = df[col].fillna(pd.Timestamp('1970-01-01'))
    else:
        df[col] = df[col].fillna(0)

#### FASE 2: Segmentação de Clientes (K-Means)

In [50]:
# Aplicar a clusterização no DataFrame 'df' completo antes de filtrar para modelagem.
print("FASE 2: Segmentando clientes com K-Means...")

features_cluster = ['vl_total_gasto', 'qtd_compras', 'cd_faturamento']
X_cluster = df[features_cluster]

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

FASE 2: Segmentando clientes com K-Means...


In [51]:
# Adiciona a coluna 'segmento_cliente' ao DataFrame principal 'df'
df['segmento_cliente'] = kmeans.fit_predict(X_cluster_scaled)

print("Análise dos Segmentos de Clientes (Médias por cluster):")
print(df.groupby('segmento_cliente')[features_cluster].mean())
print("\n" + "="*50 + "\n")

Análise dos Segmentos de Clientes (Médias por cluster):
                  vl_total_gasto  qtd_compras  cd_faturamento
segmento_cliente                                             
0                   3.645057e+05   123.108077       42.463026
1                   4.949348e+05   111.710838       50.835610
2                   1.432378e+07  2090.119565       47.304348
3                   1.242304e+08  4411.600000       49.400000




#### FASE 1.5: Preparação final para modelagem

In [52]:
# Agora, filtramos o DataFrame 'df' (que já tem o segmento) para criar a base de modelagem.
def classificar_nps(nota):
    if nota <= 6:
        return 'Detrator'
    elif nota <= 8:
        return 'Neutro'
    else: # nota > 8
        return 'Promotor'

df_com_nps = df[df['resposta_NPS'] != 0].copy()
df_com_nps['categoria_nps'] = df_com_nps['resposta_NPS'].apply(classificar_nps)
df_modelagem = df_com_nps.copy()

print("DataFrame consolidado e preparado para modelagem:")
print(df_modelagem[['cd_cliente', 'ds_segmento', 'vl_total_gasto', 'categoria_nps', 'segmento_cliente']].head())
print(f"Total de clientes com consumo e NPS para modelagem: {len(df_modelagem)}")
print("\n" + "="*50 + "\n")

DataFrame consolidado e preparado para modelagem:
  cd_cliente            ds_segmento  vl_total_gasto categoria_nps  \
0     T00053             MANUFATURA   254759.053356      Promotor   
1     T00082  CONSTRUCAO E PROJETOS   452679.096475      Promotor   
2     T00145              LOGISTICA      728.486443      Promotor   
3     T00336             MANUFATURA   465034.556976        Neutro   
4     T00673             MANUFATURA   183124.075261      Promotor   

   segmento_cliente  
0                 0  
1                 0  
2                 0  
3                 1  
4                 0  
Total de clientes com consumo e NPS para modelagem: 2175




#### FASE 3: Modelo Preditivo de NPS

In [53]:
print("FASE 3: Treinando modelo para prever a categoria NPS...")

features_modelo = [
    'vl_total_gasto', 'qtd_compras', 'vl_total_desconto', 
    'ds_segmento', 'faixa_faturamento', 'segmento_cliente',
    'situacao_contrato', 'modal_comerc'
]
target_modelo = 'categoria_nps'

X = df_modelagem[features_modelo]
y = df_modelagem[target_modelo]

categorical_features = ['ds_segmento', 'faixa_faturamento', 'situacao_contrato', 'modal_comerc']
numerical_features = ['vl_total_gasto', 'qtd_compras', 'vl_total_desconto']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

FASE 3: Treinando modelo para prever a categoria NPS...


In [54]:
# Adicionar 'class_weight' para lidar com o desbalanceamento de classes
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced'))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [55]:
print("Relatório de Classificação do Modelo Preditivo:")
print(classification_report(y_test, y_pred))
print("\n" + "="*50 + "\n")

print("FASE 3.1: Análise dos fatores mais influentes no NPS...")
feature_names_raw = model.named_steps['preprocessor'].get_feature_names_out()
feature_names = [name.split('__')[-1] for name in feature_names_raw]
importances = model.named_steps['classifier'].feature_importances_
feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
print("Principais Variáveis que Influenciam o NPS:")
print(feature_importance_df.head(10))
print("\n" + "="*50 + "\n")

Relatório de Classificação do Modelo Preditivo:
              precision    recall  f1-score   support

    Detrator       0.20      0.02      0.04        41
      Neutro       0.43      0.38      0.40       175
    Promotor       0.51      0.64      0.57       219

    accuracy                           0.48       435
   macro avg       0.38      0.35      0.34       435
weighted avg       0.45      0.48      0.45       435



FASE 3.1: Análise dos fatores mais influentes no NPS...
Principais Variáveis que Influenciam o NPS:
                                            feature  importance
0                                    vl_total_gasto    0.218667
1                                       qtd_compras    0.209227
2                                 vl_total_desconto    0.122844
10                           ds_segmento_MANUFATURA    0.024958
38              modal_comerc_MODALIDADE TRADICIONAL    0.024384
40                                 segmento_cliente    0.021884
13                   

#### FASE 4: Geração da Tabela de Previsão

In [56]:
print("FASE 4: Gerando a tabela final de previsão...")

# CORREÇÃO: Agora 'df' já tem a coluna 'segmento_cliente' e a predição funcionará.
df['nps_previsto_categoria'] = model.predict(df[features_modelo])

mapa_nota = {'Promotor': 9, 'Neutro': 8, 'Detrator': 5}
df['resposta_NPS_previsao'] = df['nps_previsto_categoria'].map(mapa_nota)

tabela_previsao = df[[
    'cd_cliente', 'vl_total_gasto', 'vl_total_desconto', 'qtd_compras',
    'faixa_faturamento', 'ds_segmento', 'modal_comerc', 'situacao_contrato',
    'resposta_NPS', 'resposta_NPS_previsao'
]].copy()

tabela_previsao.rename(columns={
    'vl_total_gasto': 'vl_total_historico', 
    'vl_total_desconto': 'vl_desconto_historico',
    'qtd_compras': 'qtd_produtos_historico', 
    'faixa_faturamento': 'fat_faixa',
    'resposta_NPS': 'nps_historico_nota'
}, inplace=True)

tabela_previsao['mes_previsao'] = (pd.to_datetime('today') + pd.DateOffset(months=1)).strftime('%Y-%m')
tabela_previsao['vl_total_previsao'] = tabela_previsao['vl_total_historico'] * 1.05 # Simulação simples
tabela_previsao['vl_desconto_previsao'] = tabela_previsao['vl_desconto_historico'] * 1.02 # Simulação simples
tabela_previsao['situacao_contrato_previsao'] = np.where(tabela_previsao['situacao_contrato'] == 'ATIVO', 'Manter Ativo', 'Risco de Churn')

print("Amostra da Tabela de Previsão para o Dashboard:")
print(tabela_previsao.head())

tabela_previsao.to_csv('dados_previsao_output.csv', index=False)
print("\nArquivo 'dados_previsao_output.csv' gerado com sucesso!")

FASE 4: Gerando a tabela final de previsão...
Amostra da Tabela de Previsão para o Dashboard:
  cd_cliente  vl_total_historico  vl_desconto_historico  \
0     T00053       254759.053356               0.000000   
1     T00082       452679.096475               0.000000   
2     T00145          728.486443               0.000000   
3     T00336       465034.556976               0.000000   
4     T00673       183124.075261          168722.441836   

   qtd_produtos_historico                      fat_faixa  \
0                     118    Faixa 05 - De 35 M ate 50 M   
1                     245    Faixa 04 - De 25 M ate 35 M   
2                      31    Faixa 03 - De 15 M ate 25 M   
3                      54  Faixa 08 - De 150 M ate 300 M   
4                     108    Faixa 03 - De 15 M ate 25 M   

             ds_segmento                        modal_comerc  \
0             MANUFATURA              MODALIDADE TRADICIONAL   
1  CONSTRUCAO E PROJETOS              MODALIDADE TRADICIONAL  